# E5-Style Sentence Transformer Finetuning Pipeline (Pre-computed Datasets)

## Training Configuration (E5 Paper Table 11)
- **Model**: `intfloat/e5-base-unsupervised` with mean pooling
- **Datasets**: MS-MARCO (486K) + NQ (307K) with 7 hard negatives
- **Pre-computed Scores**: Probability distributions from BAAI/bge-reranker-v2-m3 (Ï„=2.0)
- **Loss**: KL Divergence + Î± Ã— Contrastive Loss (Î±=0.2)
- **Logging**: WandB every 500 steps
- **Checkpoints**: Google Drive for crash recovery

## Key Change: Using Pre-computed HuggingFace Datasets
- NQ: `satyam2025/nq-bge-reranker-v2-m3-7hn-temperature2-softmax` (307K rows)
- MS-MARCO: `satyam2025/msmarco-bge-reranker-v2-m3-7hn-temperature2-softmax` (486K rows)

Both datasets contain pre-computed probability scores (softmax applied) that sum to 1.0.

In [ ]:
#teacher temperature must be same as student temperature in the KL Divergence distillation
#Also the precision is important here. Like we need to look if values like 0.000005 are not getting truncated in the training. Although higher Temperature values lead to smoother distributions
#We need to use  the 7 HNs inboth KL nad in the in-batch negatives loss calculation & training.

In [ ]:
# ============================================================================
# CELL 1: Environment Configuration & Memory Optimization
# ============================================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Enable TF32 for faster matrix multiplications on A100
import torch
# Optimized: Enable TF32 with new API
torch.set_float32_matmul_precision('high')


print("âœ… Memory configuration set")
print(f"   PYTORCH_CUDA_ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'Not set')}")
print(f"   TF32 Precision: {torch.get_float32_matmul_precision()}")

In [ ]:
# ============================================================================
# CELL 2: Install Dependencies
# ============================================================================
import subprocess
import sys

packages = [
    'transformers>=4.46.0',
    'datasets>=3.0.0',
    'sentence-transformers>=3.3.0',
    'accelerate>=1.2.0',
    'flash-attn>=2.0.0 --no-build-isolation',
    'wandb>=0.16.0',
    'torch>=2.0.0',
    'tqdm>=4.66.0',
    'numpy>=1.24.0',
    'huggingface-hub>=0.20.0',
]

print("Installing dependencies...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f"  âœ“ {package}")
    except Exception as e:
        print(f"  âœ— Warning: Could not install {package}: {e}")

print("\nâœ… Dependencies installed")

In [ ]:
# ============================================================================
# CELL 3: Imports
# ============================================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import random
import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Transformer, Pooling
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

import wandb
from huggingface_hub import HfApi, login as hf_login

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================================
# CELL 4: Configuration - E5 Paper Hyperparameters
# ============================================================================
@dataclass
class E5Config:
    """E5 Paper Table 11 Hyperparameters"""
    # Model
    model_name: str = "intfloat/e5-base-unsupervised"
    teacher_model: str = "BAAI/bge-reranker-v2-m3"  # Used to generate pre-computed scores
    max_length: int = 192

    # Training
    learning_rate: float = 2e-5
    warmup_steps: int = 400
    batch_size: int = 256
    gradient_accumulation_steps: int = 1  # No accumulation needed (High VRAM)
    per_device_batch_size: int = 256
    epochs: int = 3
    weight_decay: float = 0.01

    # Loss
    kl_temperature: float = 2.0  # Temperature for KL divergence (matches teacher softmax Ï„=2.0)
    contrastive_temperature: float = 0.01  # Temperature for InfoNCE (sharp discrimination)
    alpha: float = 0.2  # Contrastive loss weight
    num_hard_negatives: int = 7

    # Logging & Checkpoints
    logging_steps: int = 500
    save_steps: int = 500
    eval_steps: int = 500

    # Credentials
    wandb_api_key: str = "YOUR_WANDB_API_KEY"
    hf_token: str = "YOUR_HF_TOKEN"
    hf_username: str = "satyam2025"

    # Pre-computed dataset repos
    nq_dataset_repo: str = "satyam2025/nq-bge-reranker-v2-m3-7hn-temperature2-softmax"
    msmarco_dataset_repo: str = "satyam2025/msmarco-bge-reranker-v2-m3-7hn-temperature2-softmax"

config = E5Config()

print("="*60)
print("E5 FINETUNING CONFIGURATION")
print("="*60)
print(f"Model: {config.model_name}")
print(f"Teacher (scores from): {config.teacher_model}")
print(f"Learning Rate: {config.learning_rate}")
print(f"Warmup Steps: {config.warmup_steps}")
print(f"Batch Size: {config.batch_size}")
print(f"Max Length: {config.max_length}")
print(f"Epochs: {config.epochs}")
print(f"KL Temperature: {config.kl_temperature}")
print(f"Contrastive Temperature: {config.contrastive_temperature}")
print(f"Alpha (Î±): {config.alpha}")
print(f"Hard Negatives: {config.num_hard_negatives}")
print("="*60)
print(f"\nðŸ“¦ Pre-computed Datasets:")
print(f"   NQ: {config.nq_dataset_repo}")
print(f"   MS-MARCO: {config.msmarco_dataset_repo}")
print("="*60)

In [ ]:
# ============================================================================
# CELL 5: Mount Google Drive & Setup Paths
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

# Create dated folder structure
DATE = datetime.date.today().strftime("%Y-%m-%d")
BASE_PATH = Path(f"/content/drive/MyDrive/NQ_full_INFO-KD-train-e5-klD-{DATE}")

# Directory structure
PATHS = {
    'checkpoints': BASE_PATH / 'checkpoints',
    'datasets': BASE_PATH / 'datasets',
    'models': BASE_PATH / 'models',
    'logs': BASE_PATH / 'logs',
    'cache': BASE_PATH / 'cache',
}

for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"âœ… Created: {path}")

print(f"\nðŸ“ Base path: {BASE_PATH}")

In [ ]:
# ============================================================================
# CELL 6: Load Pre-computed NQ Dataset from HuggingFace
# ============================================================================
from datasets import load_dataset

print("ðŸ“¥ Loading Pre-computed NQ Dataset from HuggingFace...")
print(f"   Repository: {config.nq_dataset_repo}")

try:
    # Load dataset directly
    nq_dataset = load_dataset(config.nq_dataset_repo, split='train')
    print(f"âœ… Loaded NQ Dataset: {len(nq_dataset):,} queries")

    # Convert to list of dicts for compatibility with existing pipeline code
    print("ðŸ”„ Converting to list format...")
    nq_scored_data = list(nq_dataset)
    print(f"âœ… Ready for training: {len(nq_scored_data):,} examples")

except Exception as e:
    print(f"âŒ Failed to load NQ dataset: {e}")
    raise e

In [ ]:
# ============================================================================
# CELL 7: Load Pre-computed MS-MARCO Dataset from HuggingFace
# ============================================================================
print("ðŸ“¥ Loading Pre-computed MS-MARCO Dataset from HuggingFace...")
print(f"   Repository: {config.msmarco_dataset_repo}")

try:
    # Load dataset directly
    msmarco_dataset = load_dataset(config.msmarco_dataset_repo, split='train')
    print(f"âœ… Loaded MS-MARCO Dataset: {len(msmarco_dataset):,} queries")

    # Convert to list of dicts for compatibility
    print("ðŸ”„ Converting to list format...")
    msmarco_scored_data = list(msmarco_dataset)
    print(f"âœ… Ready for training: {len(msmarco_scored_data):,} examples")

except Exception as e:
    print(f"âŒ Failed to load MS-MARCO dataset: {e}")
    raise e

In [ ]:
# ============================================================================
# VERIFICATION CELL 1: NQ Dataset with Pre-computed Teacher Scores
# ============================================================================
print("="*80)
print("ðŸ” VERIFICATION: NQ DATASET (With Pre-computed Teacher Scores)")
print("="*80)

print(f"\nðŸ“Š Total scored queries: {len(nq_scored_data):,}")

# Verify score sums (scores should sum to 1.0 as they are probabilities)
score_sums = []
for item in nq_scored_data[:100]:  # Check first 100
    total = sum(item['positives']['score']) + sum(item['negatives']['score'])
    score_sums.append(total)

print(f"\nðŸ“ˆ Score sum verification (first 100):")
print(f"   Min sum: {min(score_sums):.6f}")
print(f"   Max sum: {max(score_sums):.6f}")
print(f"   All sums equal 1.0: {all(abs(s - 1.0) < 0.001 for s in score_sums)}")

# Show examples
for idx in range(3):
    item = nq_scored_data[idx]
    print(f"\n{'â”€'*80}")
    print(f"ðŸ“Œ EXAMPLE {idx + 1}")
    print(f"{'â”€'*80}")
    print(f"\nðŸ”¹ QUERY: {item['query'][:100]}...")
    print(f"\nðŸ”¹ POSITIVE:")
    print(f"   Score: {item['positives']['score'][0]:.4f}")
    print(f"   Passage: {item['positives']['passage'][0][:100]}...")
    print(f"\nðŸ”¹ NEGATIVES ({len(item['negatives']['score'])} total):")
    for i, (score, passage) in enumerate(zip(item['negatives']['score'][:3],
                                             item['negatives']['passage'][:3])):
        print(f"   [{i+1}] Score: {score:.4f} | {passage[:60]}...")

    total = sum(item['positives']['score']) + sum(item['negatives']['score'])
    print(f"\n   ðŸ“Š Score sum: {total:.6f}")

print(f"\n{'='*80}")
print("âœ… NQ VERIFICATION COMPLETE")
print(f"{'='*80}")

In [ ]:
# ============================================================================
# VERIFICATION CELL 2: MS-MARCO Dataset with Pre-computed Teacher Scores
# ============================================================================
print("="*80)
print("ðŸ” VERIFICATION: MS-MARCO DATASET (With Pre-computed Teacher Scores)")
print("="*80)

print(f"\nðŸ“Š Total scored queries: {len(msmarco_scored_data):,}")

# Verify score sums (scores should sum to 1.0 as they are probabilities)
score_sums = []
for item in msmarco_scored_data[:100]:  # Check first 100
    total = sum(item['positives']['score']) + sum(item['negatives']['score'])
    score_sums.append(total)

print(f"\nðŸ“ˆ Score sum verification (first 100):")
print(f"   Min sum: {min(score_sums):.6f}")
print(f"   Max sum: {max(score_sums):.6f}")
print(f"   All sums equal 1.0: {all(abs(s - 1.0) < 0.001 for s in score_sums)}")

# Show examples
for idx in range(3):
    item = msmarco_scored_data[idx]
    print(f"\n{'â”€'*80}")
    print(f"ðŸ“Œ EXAMPLE {idx + 1}")
    print(f"{'â”€'*80}")
    print(f"\nðŸ”¹ QUERY: {item['query'][:100]}...")
    print(f"\nðŸ”¹ POSITIVE:")
    print(f"   Score: {item['positives']['score'][0]:.4f}")
    print(f"   Passage: {item['positives']['passage'][0][:100]}...")
    print(f"\nðŸ”¹ NEGATIVES ({len(item['negatives']['score'])} total):")
    for i, (score, passage) in enumerate(zip(item['negatives']['score'][:3],
                                             item['negatives']['passage'][:3])):
        print(f"   [{i+1}] Score: {score:.4f} | {passage[:60]}...")

    total = sum(item['positives']['score']) + sum(item['negatives']['score'])
    print(f"\n   ðŸ“Š Score sum: {total:.6f}")

print(f"\n{'='*80}")
print("âœ… MS-MARCO VERIFICATION COMPLETE")
print(f"{'='*80}")

In [ ]:
# ============================================================================
# CELL 10: Custom E5 Hybrid Loss Function (KL Divergence + Contrastive)
# ============================================================================
# FIXED: Pad all sequences to same max_length before concatenation
# ============================================================================

from sentence_transformers import SentenceTransformer
from torch import Tensor
from typing import Iterable, Dict

class OptimizedE5HybridLoss(nn.Module):
    """Optimized loss with batched encoding for pre-computed teacher scores"""

    def __init__(self, model, kl_temperature=1.0, contrastive_temperature=0.01, alpha=0.2, max_length=256):
        super().__init__()
        self.model = model
        self.kl_temperature = kl_temperature  # For matching teacher distribution
        self.contrastive_temperature = contrastive_temperature  # For in-batch negatives
        self.alpha = alpha
        self.max_length = max_length
        self.cross_entropy = nn.CrossEntropyLoss()

    def _pad_to_max_length(self, tensor, max_len, pad_value=0):
        """Pad tensor to max_length along dimension 1"""
        current_len = tensor.size(1)
        if current_len >= max_len:
            return tensor[:, :max_len]
        padding = torch.zeros(tensor.size(0), max_len - current_len,
                             dtype=tensor.dtype, device=tensor.device)
        if pad_value != 0:
            padding = padding.fill_(pad_value)
        return torch.cat([tensor, padding], dim=1)

    def forward(self, sentence_features, labels=None):
        batch_size = len(sentence_features[0]['input_ids'])
        device = sentence_features[0]['input_ids'].device

        # Find the max length across all features in this batch
        max_len = max(sf['input_ids'].size(1) for sf in sentence_features)

        # Pad all tensors to the same length before concatenating
        padded_input_ids = []
        padded_attention_masks = []

        for sf in sentence_features:
            # Pad input_ids (pad with 0 = [PAD] token)
            padded_ids = self._pad_to_max_length(sf['input_ids'], max_len, pad_value=0)
            padded_input_ids.append(padded_ids)

            # Pad attention_mask (pad with 0 = no attention)
            padded_mask = self._pad_to_max_length(sf['attention_mask'], max_len, pad_value=0)
            padded_attention_masks.append(padded_mask)

        # Now concatenate (all have same shape)
        all_input_ids = torch.cat(padded_input_ids, dim=0)
        all_attention_mask = torch.cat(padded_attention_masks, dim=0)

        # Single forward pass for ALL (batch_size * 9) sequences
        all_features = {
            'input_ids': all_input_ids,
            'attention_mask': all_attention_mask
        }
        all_embeddings = self.model(all_features)['sentence_embedding']
        all_embeddings = F.normalize(all_embeddings, p=2, dim=-1)

        # Split back into query and candidates
        query_emb = all_embeddings[:batch_size]  # First batch_size are queries

        # Reshape candidates: [batch_size, 8, dim]
        # The dataloader stacks: [queries, pos, neg1, neg2... neg7]
        # So we have 8 blocks of 'batch_size' candidates
        candidate_blocks = all_embeddings[batch_size:]
        # view as [8, batch, dim] then transpose to [batch, 8, dim]
        candidate_embs = candidate_blocks.view(8, batch_size, -1).transpose(0, 1)

        # ----------------------------------------------------------------------
        # 1. KL Divergence Loss (Teacher Distillation on Hard Negatives)
        # ----------------------------------------------------------------------
        # Compute similarities for hard negatives: [batch_size, 8]
        hard_neg_scores = torch.bmm(
            query_emb.unsqueeze(1),
            candidate_embs.transpose(1, 2)
        ).squeeze(1) / self.kl_temperature  # Ï„=1.0 to match teacher (Hinton approach)

        # Student distribution (log softmax for KL div)
        p_student = F.log_softmax(hard_neg_scores, dim=1)

        # KL Divergence Loss (labels are pre-computed teacher probabilities)
        if labels is not None:
            kl_loss = F.kl_div(p_student, labels, reduction='batchmean')
        else:
            kl_loss = torch.tensor(0.0, device=device)

        # ----------------------------------------------------------------------
        # 2. In-Batch Negatives Loss (Contrastive with BOTH in-batch AND hard negatives)
        # ----------------------------------------------------------------------
        # For each query, negatives = 7 hard negatives + (batch_size - 1) in-batch negatives

        # Get all hard negatives for each query: [batch_size, 7, dim]
        hard_negs = candidate_embs[:, 1:, :]  # Skip position 0 (positive)

        # Get all positives in the batch: [batch_size, dim]
        all_positives = candidate_embs[:, 0, :]

        # Build the full candidate set for each query
        # For query i: candidates = [pos_i, hard_neg_i_1..7, pos_j for j != i]

        # Method: Compute scores separately then concatenate
        # 1. Score against own positive and hard negatives: [batch, 8]
        own_candidate_scores = torch.bmm(
            query_emb.unsqueeze(1),
            candidate_embs.transpose(1, 2)
        ).squeeze(1) / self.contrastive_temperature

        # 2. Score against all other positives (in-batch): [batch, batch]
        in_batch_scores = torch.matmul(query_emb, all_positives.T) / self.contrastive_temperature

        # Create mask to exclude own positive from in-batch (already included in own_candidate_scores)
        # We'll handle this by setting diagonal to -inf
        mask = torch.eye(batch_size, device=device, dtype=torch.bool)
        in_batch_scores_masked = in_batch_scores.masked_fill(mask, float('-inf'))

        # Concatenate: [batch, 8 + batch_size]
        # Position 0 = own positive, positions 1-7 = hard negatives, positions 8+ = in-batch negatives
        contrastive_scores = torch.cat([own_candidate_scores, in_batch_scores_masked], dim=1)

        # The correct positive for query i is at index 0
        contrastive_labels = torch.zeros(batch_size, device=device, dtype=torch.long)

        # Contrastive Loss (Combined In-Batch + Hard Negatives)
        in_batch_loss = self.cross_entropy(contrastive_scores, contrastive_labels)

        # ----------------------------------------------------------------------
        # Total Loss
        # ----------------------------------------------------------------------
        return kl_loss + self.alpha * in_batch_loss
    def get_config_dict(self):
        return {'kl_temperature': self.kl_temperature, 'contrastive_temperature': self.contrastive_temperature, 'alpha': self.alpha}

print("âœ… E5HybridLoss class defined (with padding fix)")
print("\nðŸ“ Loss Formula:")
print("   L = D_KL(p_teacher || p_student) + Î± Ã— L_cont")
print(f"   - Î± = {config.alpha}")
print(f"   - Ï„_KL = {config.kl_temperature} (matches teacher softmax Ï„=1.0)")
print(f"   - Ï„_cont = {config.contrastive_temperature} (sharp for InfoNCE)")
print("\nðŸ”§ FIX: All sequences are now padded to same length before concatenation")

In [ ]:
# ============================================================================
# CELL 11: Prepare Combined Training Dataset
# ============================================================================
# Both datasets have the same structure:
# - query: string
# - positives: {passage: [str], score: [float]}
# - negatives: {passage: [str, ...], score: [float, ...]}
#
# Scores are already probability distributions (sum to 1.0)
# ============================================================================

def prepare_training_dataset(nq_data, msmarco_data, config):
    """Prepare combined training dataset using pre-computed probability scores"""
    print("\n" + "="*80)
    print("PREPARING COMBINED TRAINING DATASET")
    print("="*80)

    all_examples = []

    # ==================== NQ ====================
    print("Processing NQ dataset...")
    for item in tqdm(nq_data, desc="NQ"):
        # Combine positive and negative scores into a single list (already probabilities)
        teacher_scores = [item['positives']['score'][0]] + list(item['negatives']['score'])

        example = {
            'anchor': "query: " + item['query'],
            'positive': "passage: " + item['positives']['passage'][0],
            'label': teacher_scores  # â† FIXED: Changed from 'teacher_scores' to 'label'
        }
        for i, neg_passage in enumerate(item['negatives']['passage'], 1):
            example[f'negative_{i}'] = "passage: " + neg_passage
        all_examples.append(example)

    # ==================== MS-MARCO ====================
    print("Processing MS-MARCO dataset...")
    for item in tqdm(msmarco_data, desc="MS-MARCO"):
        # Combine positive and negative scores into a single list (already probabilities)
        teacher_scores = [item['positives']['score'][0]] + list(item['negatives']['score'])

        example = {
            'anchor': "query: " + item['query'],
            'positive': "passage: " + item['positives']['passage'][0],
            'label': teacher_scores  # â† FIXED: Changed from 'teacher_scores' to 'label'
        }
        for i, neg_passage in enumerate(item['negatives']['passage'], 1):
            example[f'negative_{i}'] = "passage: " + neg_passage
        all_examples.append(example)

    print(f"\nðŸ“Š Total combined examples: {len(all_examples):,}")
    print(f"   - NQ: {len(nq_data):,}")
    print(f"   - MS-MARCO: {len(msmarco_data):,}")

    # Create dataset
    combined_dataset = Dataset.from_list(all_examples)
    combined_dataset = combined_dataset.shuffle(seed=42)
    split = combined_dataset.train_test_split(test_size=0.01, seed=42)

    print(f"\nâœ… Dataset split:")
    print(f"   Train: {len(split['train']):,}")
    print(f"   Eval: {len(split['test']):,}")

    return split

# Prepare dataset
train_eval_split = prepare_training_dataset(nq_scored_data, msmarco_scored_data, config)
train_dataset = train_eval_split['train']
eval_dataset = train_eval_split['test']

print(f"\n{'='*80}")
print("âœ… DATASET PREPARATION COMPLETE")
print(f"{'='*80}")

In [ ]:
# ============================================================================
# CELL 12: WandB Setup and Login
# ============================================================================
print("ðŸ” Setting up WandB...")

# Login to WandB
wandb.login(key=config.wandb_api_key)

# Initialize WandB run
wandb.init(
    project="e5-finetuning-kl-distillation",
    name=f"e5-base-precomputed-{DATE}",
    config={
        'model': config.model_name,
        'teacher': config.teacher_model,
        'learning_rate': config.learning_rate,
        'warmup_steps': config.warmup_steps,
        'batch_size': config.batch_size,
        'epochs': config.epochs,
        'max_length': config.max_length,
        'kl_temperature': config.kl_temperature,
        'contrastive_temperature': config.contrastive_temperature,
        'alpha': config.alpha,
        'num_hard_negatives': config.num_hard_negatives,
        'weight_decay': config.weight_decay,
        'nq_dataset': config.nq_dataset_repo,
        'msmarco_dataset': config.msmarco_dataset_repo,
    },
    tags=['e5', 'kl-divergence', 'contrastive', 'nq', 'msmarco', 'precomputed-scores'],
)

print(f"\nâœ… WandB initialized")
print(f"   Project: e5-finetuning-kl-distillation")
print(f"   Run: e5-base-precomputed-{DATE}")
print(f"   Dashboard: https://wandb.ai/your-username/e5-finetuning-kl-distillation")

In [ ]:
# ============================================================================
# CELL 13: Training Configuration and Trainer Setup
# ============================================================================

# Create E5 model with mean pooling
print("ðŸ“¥ Loading E5 model...")
transformer = Transformer(config.model_name, max_seq_length=config.max_length)
pooling = Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling])

# Enable gradient checkpointing for memory efficiency (Recommended for batch size 256)
model[0].auto_model.gradient_checkpointing_enable()
print("âœ… Gradient checkpointing enabled")

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nðŸ“Š Model Info:")
print(f"   Name: {config.model_name}")
print(f"   Pooling: Mean")
print(f"   Parameters: {total_params:,}")
print(f"   Trainable: {trainable_params:,}")
print(f"   Embedding dim: {model.get_sentence_embedding_dimension()}")

# Create loss function
loss_fn = OptimizedE5HybridLoss(
    model=model,
    kl_temperature=config.kl_temperature,
    contrastive_temperature=config.contrastive_temperature,
    alpha=config.alpha
)

# Training arguments
MODEL_OUTPUT_PATH = PATHS['models'] / "e5-finetuned"

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(MODEL_OUTPUT_PATH),
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.per_device_batch_size,
    per_device_eval_batch_size=config.per_device_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,

    # OPTIMIZED DataLoader Settings
    dataloader_num_workers=8,  # Increased for A100
    dataloader_prefetch_factor=2,  # Reduced to avoid memory issues
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,

    # Logging
    logging_steps=config.logging_steps,
    logging_first_step=True,

    # Evaluation
    eval_strategy="epoch",

    # Checkpointing
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Optimization
    bf16=True,
    tf32=True,
    fp16=False,
    group_by_length=False,  # DISABLE - causes slowdown with variable lengths

    # WandB
    report_to="wandb",
    run_name=f"e5-base-precomputed-{DATE}",

    seed=42,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    disable_tqdm=False,
)

print(f"\n{'='*60}")
print("TRAINING CONFIGURATION")
print(f"{'='*60}")
print(f"Output: {MODEL_OUTPUT_PATH}")
print(f"Batch Size: {config.per_device_batch_size}")
print(f"Learning Rate: {config.learning_rate}")
print(f"Warmup Steps: {config.warmup_steps}")
print(f"Epochs: {config.epochs}")
print(f"KL Temperature: {config.kl_temperature}")
print(f"Contrastive Temperature: {config.contrastive_temperature}")
print(f"Alpha: {config.alpha}")
print(f"{'='*60}")

In [ ]:
# ============================================================================
# CELL 13.5: Compare Teacher vs Student Scores (Before Training)
# ============================================================================
# This cell compares:
# - Teacher scores: Pre-computed probabilities (T=2 softmax) from the dataset
# - Student scores: Computed from current E5 model embeddings
# ============================================================================

print("=" * 80)
print("ðŸ” TEACHER vs STUDENT SCORE COMPARISON (Before Training)")
print("=" * 80)

def compare_scores(dataset_name, scored_data, model, tokenizer, num_examples=5):
    """Compare teacher and student scores for first N examples"""
    print(f"\n{'â”€'*80}")
    print(f"ðŸ“Š {dataset_name} Dataset - First {num_examples} Examples")
    print(f"{'â”€'*80}")

    for idx in range(min(num_examples, len(scored_data))):
        item = scored_data[idx]

        # Get teacher scores (pre-computed probabilities)
        teacher_pos_score = item['positives']['score'][0]
        teacher_neg_scores = item['negatives']['score']
        teacher_all = [teacher_pos_score] + list(teacher_neg_scores)

        # Prepare texts for encoding
        query = "query: " + item['query']
        positive = "passage: " + item['positives']['passage'][0]
        negatives = ["passage: " + neg for neg in item['negatives']['passage']]

        # Encode with student model
        with torch.no_grad():
            q_emb = model.encode([query], normalize_embeddings=True)
            p_emb = model.encode([positive], normalize_embeddings=True)
            n_embs = model.encode(negatives, normalize_embeddings=True)

        # Compute student similarities (cosine)
        student_pos_sim = float(np.dot(q_emb[0], p_emb[0]))
        student_neg_sims = [float(np.dot(q_emb[0], n_emb)) for n_emb in n_embs]
        student_all_sims = [student_pos_sim] + student_neg_sims

        # Convert student similarities to probabilities using T=2 (to match teacher)
        student_logits = torch.tensor(student_all_sims) / config.kl_temperature
        student_probs = F.softmax(student_logits, dim=0).tolist()

        # Print comparison
        print(f"\nðŸ“Œ Example {idx + 1}: {item['query'][:60]}...")
        print(f"   {'Candidate':<12} {'Teacher Prob':>14} {'Student Sim':>14} {'Student Prob':>14}")
        print(f"   {'-'*56}")

        labels = ["Positive"] + [f"Negative {i+1}" for i in range(len(teacher_neg_scores))]
        for i, label in enumerate(labels):
            print(f"   {label:<12} {teacher_all[i]:>14.6f} {student_all_sims[i]:>14.4f} {student_probs[i]:>14.6f}")

        # Summary stats
        teacher_pos_rank = 1 + sum(1 for s in teacher_neg_scores if s > teacher_pos_score)
        student_pos_rank = 1 + sum(1 for s in student_neg_sims if s > student_pos_sim)
        print(f"   âž¡ï¸ Teacher ranks Positive: #{teacher_pos_rank} | Student ranks Positive: #{student_pos_rank}")

# Compare NQ samples
compare_scores("NQ", nq_scored_data, model, None, num_examples=5)

# Compare MSMARCO samples
compare_scores("MS-MARCO", msmarco_scored_data, model, None, num_examples=5)

print(f"\n{'='*80}")
print("âœ… Score comparison complete. Proceed with training to align student to teacher.")
print(f"{'='*80}")


In [ ]:
# ============================================================================
# CELL 14: Run Training
# ============================================================================

# Create trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss_fn,
)

# Calculate training statistics
steps_per_epoch = len(train_dataset) // config.batch_size
total_steps = steps_per_epoch * config.epochs

print("\n" + "="*80)
print("ðŸš€ STARTING TRAINING")
print("="*80)
print(f"\nðŸ“Š Training Statistics:")
print(f"   Training examples: {len(train_dataset):,}")
print(f"   Validation examples: {len(eval_dataset):,}")
print(f"   Steps per epoch: {steps_per_epoch:,}")
print(f"   Total steps: {total_steps:,}")
print(f"   Estimated time: {total_steps * 2 / 3600:.1f} hours (assuming ~2s/step)")
print("\nâš ï¸ Checkpoints are saved to Google Drive every epoch.")
print("   If training crashes, re-run this cell to resume from last checkpoint.")
print("\n" + "-"*80)

# Start training
print("\nðŸ†• Starting fresh training...")
train_result = trainer.train()

# Training complete
print("\n" + "="*80)
print("âœ… TRAINING COMPLETE")
print("="*80)
print(f"\nðŸ“Š Final Metrics:")
for key, value in train_result.metrics.items():
    print(f"   {key}: {value}")

# Save final model
FINAL_MODEL_PATH = PATHS['models'] / "e5-finetuned-final"
model.save(str(FINAL_MODEL_PATH))
print(f"\nðŸ’¾ Final model saved to: {FINAL_MODEL_PATH}")

# Save training metadata
metadata = {
    'model_name': config.model_name,
    'training_date': DATE,
    'config': {
        'learning_rate': config.learning_rate,
        'warmup_steps': config.warmup_steps,
        'batch_size': config.batch_size,
        'epochs': config.epochs,
        'temperature': config.temperature,
        'alpha': config.alpha,
        'num_hard_negatives': config.num_hard_negatives,
    },
    'metrics': train_result.metrics,
    'datasets': {
        'nq_repo': config.nq_dataset_repo,
        'nq_size': len(nq_scored_data),
        'msmarco_repo': config.msmarco_dataset_repo,
        'msmarco_size': len(msmarco_scored_data),
    }
}

with open(FINAL_MODEL_PATH / 'training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"   Metadata saved to: {FINAL_MODEL_PATH / 'training_metadata.json'}")

# Finish WandB
wandb.finish()
print("\nðŸ WandB run finished")

In [ ]:
# ============================================================================
# CELL 15: Verify Final Model Output
# ============================================================================
print("="*80)
print("ðŸ” VERIFICATION: FINAL MODEL")
print("="*80)

# List saved files
print(f"\nðŸ“ Files in {FINAL_MODEL_PATH}:")
for f in FINAL_MODEL_PATH.iterdir():
    if f.is_file():
        print(f"   âœ… {f.name} ({f.stat().st_size:,} bytes)")
    elif f.is_dir():
        print(f"   ðŸ“ {f.name}/")

# Test model loading
print("\nðŸ”„ Testing model loading...")
test_model = SentenceTransformer(str(FINAL_MODEL_PATH))
print(f"   âœ… Model loaded successfully")
print(f"   Embedding dimension: {test_model.get_sentence_embedding_dimension()}")

# Test encoding
test_queries = [
    "query: What is machine learning?",
    "query: How does neural network work?"
]
test_passages = [
    "passage: Machine learning is a subset of artificial intelligence.",
    "passage: Neural networks are computing systems inspired by biological neural networks."
]

print("\nðŸ§ª Testing embeddings...")
q_embs = test_model.encode(test_queries, normalize_embeddings=True)
p_embs = test_model.encode(test_passages, normalize_embeddings=True)

# Compute similarities
similarities = np.dot(q_embs, p_embs.T)
print(f"\nðŸ“Š Query-Passage Similarities:")
for i, q in enumerate(test_queries):
    for j, p in enumerate(test_passages):
        print(f"   Q{i+1} <-> P{j+1}: {similarities[i,j]:.4f}")

print(f"\n{'='*80}")
print("âœ… MODEL VERIFICATION COMPLETE")
print(f"{'='*80}")

In [ ]:
from google.colab import runtime
# Terminate the session to release GPU
runtime.unassign()

In [ ]:
# # # ============================================================================
# # # CELL 16: Cleanup (Optional)
# # # ============================================================================
# import torch
# import gc

# # # Explicitly call Python's garbage collector to free references
# # gc.collect()

# # Empty the PyTorch CUDA memory cache
# torch.cuda.empty_cache()

# print("âœ… Memory cleanup complete")